# 04 - Modelado y comparacion temporal

## 1. Objetivo
- **Pregunta:** que modelos anticipan mejor uplift y ROI de promociones futuras?
- **Decision:** comparar baselines y un maximo de tres familias con validacion temporal.
- **Fundamento tecnico:** el error medio se complementa con estabilidad, segmentos, unidades y curvas.
- **Fundamento de negocio:** una recomendacion debe crear volumen sin aprobar ROI destructivo.
- **Resultado:** los champions se fijan exclusivamente con desarrollo OOF.
- **Limitacion:** esta fase es predictiva; no estima efectos causales ni optimiza promociones.

## 2. Contexto de negocio
- **Pregunta:** por que no basta minimizar una metrica?
- **Decision:** ponderar error, riesgo ROI, suavidad, explicabilidad y complejidad.
- **Fundamento tecnico:** modelos con errores similares pueden extrapolar de forma muy distinta.
- **Fundamento de negocio:** falsos positivos de ROI pueden aprobar promociones destructoras de valor.
- **Resultado:** el scorecard usa criterios 1-5 y pesos declarados.
- **Limitacion:** los pesos son una politica de decision revisable, no una verdad estadistica.

## 3. Trazabilidad del codigo
- **Pregunta:** donde vive la logica reproducible?
- **Decision:** el notebook llama una vez al pipeline y luego consume artifacts.
- **Fundamento tecnico:** splits, preprocesamiento, metricas y registry viven en `src/mini_tpo/`.
- **Fundamento de negocio:** una ruta unica reduce diferencias entre analisis y uso posterior.
- **Resultado:** tablas, modelos, predicciones y figuras se regeneran juntos.
- **Limitacion:** la fecha de entrenamiento del registry cambia en cada ejecucion.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('No se encontro la raiz relativa del proyecto')
sys.path.insert(0, str(ROOT / 'src'))

from mini_tpo.pipeline import run_modeling_comparison

outputs = run_modeling_comparison()  # unica ejecucion del pipeline
TABLES = ROOT / 'reports' / 'tables'
def table(name):
    return pd.read_csv(TABLES / name)

print(f"Desarrollo: {len(outputs['development']):,} filas")
print(f"Test final: {len(outputs['final_test']):,} filas desde {outputs['test_start'].date()}")
print(f"Version de datos: {outputs['data_version'][:12]}")


## 4. Datos y feature sets
- **Pregunta:** que representaciones aportan sin combinar todo indiscriminadamente?
- **Decision:** evaluar base, interacciones, no lineales, mensual, semanal y extended mediante ablacion Ridge.
- **Fundamento tecnico:** Ridge aisla el aporte de representacion con bajo coste.
- **Fundamento de negocio:** reduce el riesgo de justificar complejidad que no mejora promociones futuras.
- **Resultado:** el mejor set OOF se reutiliza en las tres familias por target.
- **Limitacion:** no es una busqueda exhaustiva modelo por feature set.

## 5. Anti-leakage
- **Pregunta:** que informacion era conocida al decidir la promocion?
- **Decision:** excluir targets, postpromocion, auditoria y `row_id` de features.
- **Fundamento tecnico:** encoders y escaladores se ajustan dentro de cada fold.
- **Fundamento de negocio:** evita expectativas de ROI imposibles de reproducir antes de ejecutar.
- **Resultado:** los manifests y tests validan listas de features por artifact.
- **Limitacion:** disponibilidad prepromocion de baseline y elasticidad sigue siendo supuesto del caso.

## 6. Test final
- **Pregunta:** que periodo representa promociones futuras sin perder cobertura?
- **Decision:** reservar octubre-diciembre de 2025.
- **Fundamento tecnico:** son los ultimos tres meses completos disponibles y no entran a tuning.
- **Fundamento de negocio:** conserva los 15 SKU, las 3 cadenas y las 45 combinaciones.
- **Resultado:** 357 filas aisladas; desarrollo termina el 29-09-2025.
- **Limitacion:** un trimestre no cubre todos los shocks comerciales posibles.

## 7. Splits temporales
- **Pregunta:** como medir estabilidad sin mezclar futuro y pasado?
- **Decision:** cuatro ventanas expansivas sobre fechas unicas.
- **Fundamento tecnico:** una misma `fecha_inicio_tanda` nunca se divide.
- **Fundamento de negocio:** replica el aprendizaje acumulado disponible en cada decision.
- **Resultado:** el fold 4 incluye SKU no vistos para tensionar generalizacion.
- **Limitacion:** los primeros meses solo aparecen como entrenamiento inicial.


In [ ]:
display(table('modeling_final_test_coverage.csv'))
display(table('modeling_temporal_splits.csv'))
display(table('modeling_ablation_results.csv')[['target','feature_set','mae','mae_fold_std']])


## 8. Baselines
- **Pregunta:** los modelos superan reglas historicas defendibles?
- **Decision:** mediana global, jerarquia SKU-cadena y banda de descuento para ROI.
- **Fundamento tecnico:** cada estadistico usa solo train con fallback explicito.
- **Fundamento de negocio:** el modelo debe justificar su coste frente a una regla sencilla.
- **Resultado:** los candidatos seleccionados mejoran materialmente los baselines OOF.
- **Limitacion:** los baselines no capturan tendencia ni interacciones continuas.

## 9. Justificacion de familias
- **Pregunta:** que sesgos inductivos conviene contrastar?
- **Decision:** Ridge, HistGradientBoosting y ExtraTrees.
- **Fundamento tecnico:** contrastan suavidad regularizada, boosting robusto y bagging no lineal.
- **Fundamento de negocio:** cubren explicabilidad, respuesta tabular y riesgo operativo sin sobreingenieria.
- **Resultado:** nunca se comparan mas de tres familias.
- **Limitacion:** no se evaluan librerias externas como CatBoost o LightGBM.

## 10. Preprocesamiento
- **Pregunta:** como tratar categorias nuevas y escalas distintas?
- **Decision:** one-hot y escalado para Ridge; ordinal controlado para arboles.
- **Fundamento tecnico:** `handle_unknown` evita fallos sin aprender del futuro.
- **Fundamento de negocio:** permite evaluar SKU recientes sin excluirlos.
- **Resultado:** todo el preprocesamiento forma parte del artifact recargable.
- **Limitacion:** el ordinal encoder no expresa similitud economica entre categorias.

## 11. Hiperparametros
- **Pregunta:** cuanta busqueda es razonable para 2,048 filas?
- **Decision:** grids pequenos y fundamentados dentro de CV temporal.
- **Fundamento tecnico:** se controlan regularizacion, profundidad, hojas y numero de arboles.
- **Fundamento de negocio:** limita sobreajuste y coste de mantenimiento.
- **Resultado:** cada intento registra parametros, tiempo, dispersion y decision.
- **Limitacion:** puede no contener el optimo numerico global.


In [ ]:
display(table('baseline_metrics.csv').groupby(['target','baseline'], as_index=False).mean(numeric_only=True))
search = table('model_hyperparameter_search.csv')
display(search[['target','familia','feature_set','params','mae','mae_fold_std','tiempo_entrenamiento_seg','decision']])


## 12. Metricas uplift
- **Pregunta:** como medir error relativo sin MAPE inestable?
- **Decision:** MAE, RMSE, WAPE, sMAPE, bias y R2 secundario.
- **Fundamento tecnico:** WAPE y sMAPE toleran mejor targets pequenos.
- **Fundamento de negocio:** MAE mantiene interpretacion directa del uplift.
- **Resultado:** todas se calculan en escala original, incluso tras `log1p`.
- **Limitacion:** una media global puede ocultar SKU de bajo soporte.

## 13. Metricas ROI
- **Pregunta:** como evaluar una variable negativa y de cola pesada?
- **Decision:** MAE robusta, RMSE, mediana AE, bias, Spearman y clasificacion de signo.
- **Fundamento tecnico:** no se usa MAPE ni winsorizacion irreversible.
- **Fundamento de negocio:** se priorizan falsos positivos que aprobarian perdida.
- **Resultado:** se reportan precision, recall y F1 para ROI positivo y negativo.
- **Limitacion:** el KPI oficial no puede descomponerse sin formula contable.

## 14. Resultados CV
- **Pregunta:** que candidatos generalizan mejor en desarrollo?
- **Decision:** usar OOF concatenado y dispersion de folds.
- **Fundamento tecnico:** cada fila OOF fue predicha por un modelo que no la vio.
- **Fundamento de negocio:** aproxima el rendimiento en tandas futuras.
- **Resultado:** Ridge lidera uplift; boosting robusto lidera ROI.
- **Limitacion:** los folds comparten historia de entrenamiento por diseno expansivo.

## 15. Comparacion por fold
- **Pregunta:** el promedio depende de un unico periodo?
- **Decision:** conservar metricas por fold y comparar errores por fecha.
- **Fundamento tecnico:** la desviacion temporal entra al scorecard.
- **Fundamento de negocio:** evita adoptar un modelo fragil ante cambios de mix.
- **Resultado:** la seleccion no depende de una sola ventana.
- **Limitacion:** cuatro folds ofrecen inferencia estadistica limitada.

## 16. Comparacion por SKU y cadena
- **Pregunta:** donde se concentra el riesgo de error?
- **Decision:** medir SKU, cadena, combinacion y soporte para champion/challenger.
- **Fundamento tecnico:** se reporta peor SKU y SKU reciente en seleccion.
- **Fundamento de negocio:** una media buena no protege una cuenta o producto especifico.
- **Resultado:** los segmentos quedan exportados para revision comercial.
- **Limitacion:** segmentos pequenos tienen metricas de alta varianza.


In [ ]:
display(table('cv_metrics_uplift.csv')[['familia','target_transform','mae','wape','mae_unidades','mae_fold_std']].sort_values('mae'))
display(table('cv_metrics_roi.csv')[['familia','mae','median_ae','sign_accuracy','falsos_positivos','mae_fold_std']].sort_values('mae'))
display(table('cv_metrics_by_fold.csv')[['target','fold','familia','mae','bias']])
display(table('model_statistical_comparison.csv'))


## 17. Error en unidades
- **Pregunta:** que magnitud comercial representa un error de uplift?
- **Decision:** multiplicar residual por volumen base de tanda.
- **Fundamento tecnico:** se reportan MAE, WAPE, total absoluto y bias en unidades.
- **Fundamento de negocio:** prioriza errores donde existe mayor volumen en juego.
- **Resultado:** el scorecard incorpora MAE en unidades para uplift.
- **Limitacion:** son unidades incrementales estimadas, no margen monetario.

## 18. Piso de uplift
- **Pregunta:** el piso 0.05 altera el desempeno?
- **Decision:** conservar filas y segmentar piso versus fuera del piso.
- **Fundamento tecnico:** eliminarlo cambiaria la distribucion objetivo sin evidencia.
- **Fundamento de negocio:** puede representar reglas o promociones de respuesta minima.
- **Resultado:** ambos segmentos aparecen en `cv_metrics_segmented.csv`.
- **Limitacion:** no se conoce el mecanismo que origina los valores repetidos.

## 19. Signo de ROI
- **Pregunta:** cuantas promociones destructivas se aprobarian?
- **Decision:** contar falsos positivos y darles peso en seleccion.
- **Fundamento tecnico:** el umbral de decision descriptivo es ROI cero.
- **Fundamento de negocio:** el coste de falso positivo es asimetrico.
- **Resultado:** boosting mantiene muy pocos falsos positivos OOF.
- **Limitacion:** el umbral economico real podria incluir coste de capital.

## 20. Ablation
- **Pregunta:** calendario e interacciones mejoran realmente?
- **Decision:** comparar seis sets con Ridge y escoger por MAE OOF.
- **Fundamento tecnico:** evita multiplicar todas las combinaciones con todos los modelos.
- **Fundamento de negocio:** conserva solo complejidad respaldada por evidencia temporal.
- **Resultado:** semanal para uplift y no lineal para ROI.
- **Limitacion:** el resultado de ablacion puede depender de la familia lineal.

## 21. ROI directo versus dos etapas
- **Pregunta:** uplift predicho agrega senal valida para ROI?
- **Decision:** comparar solo con uplift OOF en train y validacion.
- **Fundamento tecnico:** `uplift_real` nunca entra como predictor.
- **Fundamento de negocio:** evita un ROI artificialmente preciso e imposible de operar.
- **Resultado:** dos etapas empeora consistentemente; se conserva ROI directo.
- **Limitacion:** el primer fold no dispone de historia OOF para entrenar la segunda etapa.


In [ ]:
segmented = table('cv_metrics_segmented.csv')
display(segmented[segmented['segmento'].isin(['flag_uplift_en_piso','banda_descuento','duracion_dias'])].head(30))
display(table('roi_direct_vs_two_stage.csv'))


## 22. Incertidumbre
- **Pregunta:** que amplitud de error acompana una prediccion?
- **Decision:** usar cuantil 90% de residuos absolutos OOF.
- **Fundamento tecnico:** se reportan cobertura y ancho global, por SKU y soporte.
- **Fundamento de negocio:** una recomendacion de bajo soporte debe penalizar incertidumbre.
- **Resultado:** cobertura OOF aproximada cercana al 90%.
- **Limitacion:** es calibracion sobre los mismos residuos OOF, no conformal estricto independiente.

## 23. Curvas
- **Pregunta:** los modelos responden de forma estable al variar palancas?
- **Decision:** diagnosticar descuento 5%-40% y duraciones soportadas en tres contextos.
- **Fundamento tecnico:** se recalculan todas las features dependientes del escenario.
- **Fundamento de negocio:** un optimizador no debe explotar saltos artificiales.
- **Resultado:** se guardan curvas de buen soporte, SKU reciente y soporte bajo.
- **Limitacion:** no se seleccionan optimos ni se afirma monotonicidad causal.

## 24. Interpretabilidad
- **Pregunta:** que variables sostienen las predicciones?
- **Decision:** coeficientes estandarizados para Ridge y permutation importance para arboles.
- **Fundamento tecnico:** la importancia se calcula en la ultima validacion temporal.
- **Fundamento de negocio:** permite revisar signos y palancas con expertos comerciales.
- **Resultado:** se exportan tablas separadas por target.
- **Limitacion:** colinealidad e importancia predictiva no implican causalidad.

## 25. Scorecard
- **Pregunta:** como combinar desempeno y riesgo operativo?
- **Decision:** escala ordinal 1-5 con pesos explicitos y desempate conservador.
- **Fundamento tecnico:** MAE pesa 25%; estabilidad, segmentos, suavidad y riesgo completan la evaluacion.
- **Fundamento de negocio:** una ganancia pequena no justifica fragilidad innecesaria.
- **Resultado:** el rol se fija antes de abrir test.
- **Limitacion:** cambios de prioridades comerciales pueden requerir nuevos pesos, usando solo desarrollo.


In [ ]:
display(table('model_uncertainty_summary.csv'))
display(table('model_feature_importance_uplift.csv').head(15))
display(table('model_feature_importance_roi.csv').head(15))
display(table('model_selection_scorecard.csv')[['target','familia','feature_set','target_transform','mae','score_total','rol_seleccion']])
for name in ['buen_soporte','sku_reciente','soporte_bajo']:
    display(Image(filename=ROOT / 'reports' / 'figures' / 'modeling' / 'response_curves' / f'response_curve_{name}.png'))


## 26. Champion y challenger
- **Pregunta:** que modelos se congelan para la siguiente fase?
- **Decision:** Ridge original/log1p para uplift y dos configuraciones boosting para ROI.
- **Fundamento tecnico:** la seleccion integra CV, estabilidad, curvas y complejidad.
- **Fundamento de negocio:** mantiene un respaldo operativo por target.
- **Resultado:** cuatro artifacts se guardan y recargan con metadata.
- **Limitacion:** un challenger no debe sustituir al champion por mirar el test.

## 27. Test final
- **Pregunta:** como rinden las decisiones congeladas en el ultimo trimestre?
- **Decision:** reentrenar en todo desarrollo y evaluar una unica vez.
- **Fundamento tecnico:** no se ajustan features, parametros ni scorecard tras verlo.
- **Fundamento de negocio:** ofrece una estimacion honesta del riesgo de despliegue.
- **Resultado:** ROI mantiene alta exactitud de signo; uplift muestra empate practico entre roles.
- **Limitacion:** el challenger uplift mejora MAE de test, pero esto no autoriza seleccion retrospectiva.

## 28. Artifacts
- **Pregunta:** puede reproducirse y auditarse cada prediccion?
- **Decision:** guardar OOF, test, cuatro pipelines joblib y registry JSON.
- **Fundamento tecnico:** `row_id`, fecha, fold, familia y feature set acompanian predicciones.
- **Fundamento de negocio:** facilita revision de promociones concretas y rollback.
- **Resultado:** los tests recargan cada modelo y vuelven a predecir.
- **Limitacion:** joblib requiere versiones compatibles de Python y scikit-learn.

## 29. Limitaciones
- **Pregunta:** que no puede afirmarse con esta evidencia?
- **Decision:** separar prediccion, descripcion y causalidad.
- **Fundamento tecnico:** datos observacionales y soporte desigual restringen extrapolacion.
- **Fundamento de negocio:** ROI oficial no revela margen, canibalizacion ni coste contable.
- **Resultado:** no se recomiendan descuentos ni duraciones en esta fase.
- **Limitacion:** shocks futuros y SKU nuevos pueden degradar calibracion.

## 30. Preparacion para optimizacion
- **Pregunta:** que condiciones debe respetar el futuro optimizador?
- **Decision:** congelar modelos y restringir escenarios por soporte local.
- **Fundamento tecnico:** recalcular features, propagar incertidumbre y penalizar extrapolacion.
- **Fundamento de negocio:** balancear uplift, ROI, unidades e inversion, no maximizar un unico ratio.
- **Resultado:** la fase deja curvas diagnosticas, registry y contratos de features.
- **Limitacion:** aun no existe funcion objetivo, presupuesto ni reglas finales de recomendacion.


In [ ]:
display(table('final_test_metrics_uplift.csv'))
display(table('final_test_metrics_roi.csv'))
registry = json.loads((ROOT / 'models' / 'model_registry.json').read_text(encoding='utf-8'))
display(pd.DataFrame(registry['models'])[['target','role','model','family','feature_set','artifact','data_version']])
required = list(outputs['paths'].values())
missing = [str(path) for path in required if not Path(path).exists()]
assert not missing, f'Artifacts faltantes: {missing}'
print(f"Validacion de existencia: {len(required)} artifacts del pipeline disponibles.")
print('Optimizador construido: NO')
